In [0]:

WITH sample AS (
  SELECT id, summary FROM support.bugzilla.silver_bugs LIMIT 3
)
SELECT id, summary,
  ai_query('databricks-llama-4-maverick',
           CONCAT('In 3 words, what part of a web browser is this bug about? ', summary)) AS answer
FROM sample

In [0]:
WITH sample AS (
  SELECT id, summary FROM support.bugzilla.silver_bugs LIMIT 3
)
SELECT id, summary,
  ai_query('databricks-gpt-oss-120b',
           CONCAT('In 3 words, what part of a web browser is this bug about? ', summary)) AS answer
FROM sample

#GPT-OSS-20B

##V1

In [0]:
DECLARE OR REPLACE VARIABLE team_guide STRING DEFAULT
'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: screen readers, keyboard navigation, accessibility APIs.
- Accounts, Sync & Install: Firefox accounts, syncing between devices, installing and updating, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the interface of the browser itself: tabs, address bar, search, bookmarks, history, downloads panel, sidebar, settings page, toolbars, themes, private browsing, session restore.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, navigation, workers, service workers, storage APIs such as IndexedDB and localStorage, notifications.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, cache, proxies, WebSockets.
- New Tab & Messaging: the New Tab page, in-product messages and onboarding, experiments, remote settings.
- OS Integration: operating-system specific windows and widgets on Windows, macOS and Linux (GTK, Wayland), default browser setting, file handling, window management.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing.
- Privacy & Security: tracking protection, site permissions, certificates and TLS, sandboxing, saved passwords and logins, web authentication, cryptography.';

CREATE OR REPLACE TABLE support.bugzilla.run_gptoss20b_v1 AS
SELECT id, team AS true_team, is_urgent,
  ai_query('databricks-gpt-oss-20b',
    CONCAT(team_guide,
           '\n\nBug title: ', summary,
           '\nBug description: ', substr(description, 1, 2000),
           '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
    modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
    failOnError => false) AS response
FROM support.bugzilla.eval_set;

In [0]:
select * from support.bugzilla.run_gptoss20b_v1

In [0]:
SELECT
  COUNT(*) AS bugs,
  ROUND(AVG(CASE WHEN response.result = true_team THEN 1 ELSE 0 END) * 100, 1) AS accuracy_pct
FROM support.bugzilla.run_gptoss20b_v1

In [0]:
SELECT
  true_team,
  COUNT(*) AS bugs,
  ROUND(AVG(CASE WHEN response.result = true_team THEN 1 ELSE 0 END) * 100, 1) AS accuracy_pct
FROM support.bugzilla.run_gptoss20b_v1
GROUP BY true_team
ORDER BY accuracy_pct DESC

In [0]:
SELECT
  true_team,
  response.result AS predicted_team,
  COUNT(*) AS bugs
FROM support.bugzilla.run_gptoss20b_v1
WHERE response.result != true_team
GROUP BY true_team, response.result
ORDER BY bugs DESC

##V2

In [0]:
DECLARE OR REPLACE VARIABLE team_guide_v2 STRING DEFAULT
'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: the accessibility engine that assistive technology relies on: screen readers (NVDA, JAWS, Narrator, VoiceOver, Orca), ARIA roles and attributes, accessible names, focus and caret reporting to screen readers, high contrast and color contrast settings, accessibility tests (accname, aria WPT tests).
- Accounts, Sync & Install: Firefox accounts, syncing between devices, the installer and updater, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the general interface of the browser: tabs and tab groups, address bar, search, bookmarks, history, downloads panel, sidebar, the settings page in general, toolbars, menus, themes, private browsing windows, session restore. Use Browser UI only when no more specific team below owns the feature.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI chatbots and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, page navigation and back/forward history, window.open, iframes, workers, service workers, storage APIs such as IndexedDB, localStorage and the file system API, notifications, pointer lock, object URLs. Site compatibility bugs where a web page behaves wrongly usually belong here.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), GPU driver problems (NVIDIA, AMD, Intel), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, the network cache, proxies, WebSockets, character encoding of downloaded pages.
- New Tab & Messaging: the New Tab page and its widgets (weather, timers, stories, shortcuts, the customize sidebar), in-product messages of every kind: promotions, infobars, feature callouts, onboarding screens, what is new pages, prompts asking the user to set Firefox as default or pin it, plus experiments (Nimbus) and remote settings.
- OS Integration: operating-system specific behavior on Windows, macOS and Linux (GTK, Wayland): native scrollbars, tooltips, cursors, fullscreen, window management, the taskbar and start menu, default browser setting, file and protocol handling.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing, build system.
- Privacy & Security: tracking protection (ETP) and its panel and exceptions, query parameter stripping, fingerprinting protection (RFP), site permissions and storage access, certificates, TLS and certificate error pages, sandboxing, saved passwords and the primary password, web authentication, cryptography, anti-malware and safe browsing.
 
Routing rule: pick the team that owns the feature or the code that is broken, not the place on screen where the problem shows up. For example, a certificate error page goes to Privacy & Security, an onboarding message goes to New Tab & Messaging, and a macOS scrollbar goes to OS Integration, even though all three appear in the browser interface.';
 
CREATE OR REPLACE TABLE support.bugzilla.run_gptoss20b_v2 AS
SELECT id, team AS true_team, is_urgent,
  ai_query('databricks-gpt-oss-20b',
    CONCAT(team_guide_v2,
           '\n\nBug title: ', summary,
           '\nBug description: ', substr(description, 1, 2000),
           '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
    modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
    failOnError => false) AS response
FROM support.bugzilla.eval_set;

In [0]:
SELECT * FROM support.bugzilla.run_gptoss20b_v2

#GPT-OSS-120B

##V1

In [0]:
DECLARE OR REPLACE VARIABLE team_guide STRING DEFAULT
'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: screen readers, keyboard navigation, accessibility APIs.
- Accounts, Sync & Install: Firefox accounts, syncing between devices, installing and updating, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the interface of the browser itself: tabs, address bar, search, bookmarks, history, downloads panel, sidebar, settings page, toolbars, themes, private browsing, session restore.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, navigation, workers, service workers, storage APIs such as IndexedDB and localStorage, notifications.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, cache, proxies, WebSockets.
- New Tab & Messaging: the New Tab page, in-product messages and onboarding, experiments, remote settings.
- OS Integration: operating-system specific windows and widgets on Windows, macOS and Linux (GTK, Wayland), default browser setting, file handling, window management.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing.
- Privacy & Security: tracking protection, site permissions, certificates and TLS, sandboxing, saved passwords and logins, web authentication, cryptography.';

CREATE OR REPLACE TABLE support.bugzilla.run_gptoss120b_v1 AS
SELECT id, team AS true_team, is_urgent,
  ai_query('databricks-gpt-oss-120b',
    CONCAT(team_guide,
           '\n\nBug title: ', summary,
           '\nBug description: ', substr(description, 1, 2000),
           '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
    modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
    failOnError => false) AS response
FROM support.bugzilla.eval_set;

In [0]:
SELECT * FROM support.bugzilla.run_gptoss120b_v1

In [0]:
SELECT
  COUNT(*) AS bugs,
  ROUND(AVG(CASE WHEN response.result = true_team THEN 1 ELSE 0 END) * 100, 1) AS accuracy_pct
FROM support.bugzilla.run_gptoss120b_v1

In [0]:
SELECT
  true_team,
  COUNT(*) AS bugs,
  ROUND(AVG(CASE WHEN response.result = true_team THEN 1 ELSE 0 END) * 100, 1) AS accuracy_pct
FROM support.bugzilla.run_gptoss120b_v1
GROUP BY true_team
ORDER BY accuracy_pct DESC

In [0]:
SELECT
  true_team,
  response.result AS predicted_team,
  COUNT(*) AS bugs
FROM support.bugzilla.run_gptoss120b_v1
WHERE response.result != true_team
GROUP BY true_team, response.result
ORDER BY bugs DESC

##V2

In [0]:
-- Prompt v2: same structure as v1, with sharper team descriptions.
-- Changes from v1:
--   1. Each team description names the features it owns that LOOK like browser UI
--      (certificate error pages, tracking protection panel, promos and infobars, native scrollbars...).
--   2. A routing rule at the end: pick the team that owns the FEATURE, and use Browser UI
--      only when no more specific team fits.
 
-- Cell 1: the team guide
DECLARE OR REPLACE VARIABLE team_guide_v2 STRING DEFAULT
'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: the accessibility engine that assistive technology relies on: screen readers (NVDA, JAWS, Narrator, VoiceOver, Orca), ARIA roles and attributes, accessible names, focus and caret reporting to screen readers, high contrast and color contrast settings, accessibility tests (accname, aria WPT tests).
- Accounts, Sync & Install: Firefox accounts, syncing between devices, the installer and updater, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the general interface of the browser: tabs and tab groups, address bar, search, bookmarks, history, downloads panel, sidebar, the settings page in general, toolbars, menus, themes, private browsing windows, session restore. Use Browser UI only when no more specific team below owns the feature.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI chatbots and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, page navigation and back/forward history, window.open, iframes, workers, service workers, storage APIs such as IndexedDB, localStorage and the file system API, notifications, pointer lock, object URLs. Site compatibility bugs where a web page behaves wrongly usually belong here.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), GPU driver problems (NVIDIA, AMD, Intel), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, the network cache, proxies, WebSockets, character encoding of downloaded pages.
- New Tab & Messaging: the New Tab page and its widgets (weather, timers, stories, shortcuts, the customize sidebar), in-product messages of every kind: promotions, infobars, feature callouts, onboarding screens, what is new pages, prompts asking the user to set Firefox as default or pin it, plus experiments (Nimbus) and remote settings.
- OS Integration: operating-system specific behavior on Windows, macOS and Linux (GTK, Wayland): native scrollbars, tooltips, cursors, fullscreen, window management, the taskbar and start menu, default browser setting, file and protocol handling.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing, build system.
- Privacy & Security: tracking protection (ETP) and its panel and exceptions, query parameter stripping, fingerprinting protection (RFP), site permissions and storage access, certificates, TLS and certificate error pages, sandboxing, saved passwords and the primary password, web authentication, cryptography, anti-malware and safe browsing.
 
Routing rule: pick the team that owns the feature or the code that is broken, not the place on screen where the problem shows up. For example, a certificate error page goes to Privacy & Security, an onboarding message goes to New Tab & Messaging, and a macOS scrollbar goes to OS Integration, even though all three appear in the browser interface.';
 
-- Cell 2: run on all 280 (same as v1, just a new variable name and table name)
CREATE OR REPLACE TABLE support.bugzilla.run_gptoss120b_v2 AS
SELECT id, team AS true_team, is_urgent,
  ai_query('databricks-gpt-oss-120b',
    CONCAT(team_guide_v2,
           '\n\nBug title: ', summary,
           '\nBug description: ', substr(description, 1, 2000),
           '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
    modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
    failOnError => false) AS response
FROM support.bugzilla.eval_set;

In [0]:
SELECT * FROM support.bugzilla.run_gptoss120b_v2

#Llama 4 Maverick

In [0]:
DECLARE OR REPLACE VARIABLE team_guide STRING DEFAULT
'You route Firefox bug reports to the team that should fix them. The teams are:
- Accessibility: screen readers, keyboard navigation, accessibility APIs.
- Accounts, Sync & Install: Firefox accounts, syncing between devices, installing and updating, importing from other browsers, enterprise policies, profile backup.
- Browser UI: the interface of the browser itself: tabs, address bar, search, bookmarks, history, downloads panel, sidebar, settings page, toolbars, themes, private browsing, session restore.
- Built-in Tools & AI: PDF viewer, screenshots, page translation, built-in AI and machine learning features.
- DOM & Web APIs: how web page code talks to the browser: HTML elements, forms, events, editing, copy and paste, drag and drop, navigation, workers, service workers, storage APIs such as IndexedDB and localStorage, notifications.
- Graphics: drawing pixels on screen: GPU rendering (WebRender), canvas, WebGL, WebGPU, images, color, smooth scrolling and zooming, VR.
- JavaScript: the JavaScript engine: JIT compiler, garbage collection, WebAssembly, the JavaScript standard library, debugger API.
- Layout & CSS: how pages are laid out and styled: CSS, flexbox, grid, text and font layout, tables, form control appearance, SVG, MathML, printing.
- Media & WebRTC: audio and video playback and recording, codecs, video calls (WebRTC), Web Audio, speech.
- Networking: HTTP, DNS, cookies, cache, proxies, WebSockets.
- New Tab & Messaging: the New Tab page, in-product messages and onboarding, experiments, remote settings.
- OS Integration: operating-system specific windows and widgets on Windows, macOS and Linux (GTK, Wayland), default browser setting, file handling, window management.
- Platform Internals: core plumbing: memory use, performance, the profiler, communication between processes (IPC), XPCOM, internationalization, spell checker, preferences, fuzzing.
- Privacy & Security: tracking protection, site permissions, certificates and TLS, sandboxing, saved passwords and logins, web authentication, cryptography.';

CREATE OR REPLACE TABLE support.bugzilla.run_llama4maverick_v1 AS
SELECT id, team AS true_team, is_urgent,
  ai_query('databricks-llama-4-maverick',
    CONCAT(team_guide,
           '\n\nBug title: ', summary,
           '\nBug description: ', substr(description, 1, 2000),
           '\n\nWhich team should handle this bug? Answer with the exact team name from the list and nothing else.'),
    modelParameters => named_struct('max_tokens', 2000, 'temperature', 0.0),
    failOnError => false) AS response
FROM support.bugzilla.eval_set;

In [0]:
SELECT * FROM support.bugzilla.run_llama4maverick_v1